In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': False,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/18 19:29:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


New Spark session created successfully


In [7]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_hand

In [8]:
import pandas as pd

# alz_df_pandas = pd.read_pickle("alz_df_apr10_1355.pkl")
# cntrl_df_pandas = pd.read_pickle("cntrl_df_apr10_1355.pkl")


# alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr15_1033.pkl")
# cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr15_1033.pkl")

alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr15_2135.pkl")
cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr15_2135.pkl")


In [9]:
%%time
alz_df_spark = spark.createDataFrame(alz_df_pandas)
cntrl_df_spark = spark.createDataFrame(cntrl_df_pandas)

CPU times: user 6min 48s, sys: 13.4 s, total: 7min 2s
Wall time: 7min 9s


In [10]:
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [11]:
alz_df.show()

25/04/18 19:37:20 WARN TaskSetManager: Stage 0 contains a task of very large size (69323 KiB). The maximum recommended task size is 1000 KiB.
25/04/18 19:37:24 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 0 (TID 0): Attempting to kill Python Worker
                                                                                

+---------+-------+---------+--------+-----------+--------------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|        FeatureValue|table_type|
+---------+-------+---------+--------+-----------+--------------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|7.020328193902969E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|3.399499109946191...|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power| 0.08723169565200806|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power|0.001139140920713544|      band|
|  sub-008|   ep-0|      Fp1| custom1|      Power|4.279543645679950...|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy| 0.34805238246917725| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower| 0.01123595517128706| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power|0.001468957751058042|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|5.043480778113008E-4|      band|
|  s

# Raw Data Visualization

# Start of data processing

In [12]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [13]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [14]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [15]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(16).persist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(16).persist()

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(16).persist()


In [16]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

25/04/18 19:37:25 WARN TaskSetManager: Stage 1 contains a task of very large size (69323 KiB). The maximum recommended task size is 1000 KiB.
25/04/18 19:37:34 WARN TaskSetManager: Stage 4 contains a task of very large size (57564 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [17]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


25/04/18 19:37:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: double, C3_Beta_Power: double, C3_Delta_Power: double, C3_Theta_Power: double, C3_custom1_Power: double, C4_Alpha_Power: double, C4_Beta_Power: double, C4_Delta_Power: double, C4_Theta_Power: double, C4_custom1_Power: double, Cz_Alpha_Power: double, Cz_Beta_Power: double, Cz_Delta_Power: double, Cz_Theta_Power: double, Cz_custom1_Power: double, F3_Alpha_Power: double, F3_Beta_Power: double, F3_Delta_Power: double, F3_Theta_Power: double, F3_custom1_Power: double, F4_Alpha_Power: double, F4_Beta_Power: double, F4_Delta_Power: double, F4_Theta_Power: double, F4_custom1_Power: double, F7_Alpha_Power: double, F7_Beta_Power: double, F7_Delta_Power: double, F7_Theta_Power: double, F7_custom1_Power: double, F8_Alpha_Power: double, F8_Beta_Power: double, F8_Delta_Power: double, F8_Theta_Power: double, F8_custom1_Power: double, Fp1_Alpha_Power: double, Fp1_Beta_Power: double, Fp1_Delta_Power: double, Fp1_Theta_Power: doub

In [18]:
type(full_df)

pyspark.sql.dataframe.DataFrame

In [19]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility

In [20]:
# Split into test and train sets
test_df = full_df.filter(col("SubjectID").isin(test_subjects))
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))


# NEED TO MIN MAX AFTER PCA!, should do something line z-score , pca , then min max (optional)

In [21]:
import dimensionality_reduction
import importlib
importlib.reload(dimensionality_reduction)
from dimensionality_reduction import min_max_normalize
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]
train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)

In [22]:
pca_input_cols = feature_cols
from dimensionality_reduction import fit_pca_model

pca_model, k_val = fit_pca_model(train_norm_df, pca_input_cols, variance_target=0.95)

print(f"PCA model fitted with {k_val} components to capture 95% variance")

25/04/18 19:38:41 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/04/18 19:38:41 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
                                                                                

PCA model fitted with 22 components to capture 95% variance


In [23]:
pca_model.explainedVariance

DenseVector([0.5277, 0.1446, 0.0776, 0.0358, 0.0305, 0.0264, 0.0225, 0.0136, 0.0112, 0.01, 0.0078, 0.0071, 0.0058, 0.0047, 0.0042, 0.0039, 0.0037, 0.0034, 0.0029, 0.0028, 0.0026, 0.0024])

In [24]:
from dimensionality_reduction import apply_pca_model

train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


# ML time

In [25]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()
spark.stop()

In [26]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
X_train = np.array(train_pd["features"].tolist())
y_train = train_pd["label"].values

X_test = np.array(test_pd["features"].tolist())
y_test = test_pd["label"].values


In [27]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_train shape: (168652, 22)
y_train shape: (168652,)


In [28]:
y_train

array([1, 1, 1, ..., 0, 0, 0], dtype=int32)

In [29]:
X_train

array([[ 3.37414139,  6.6935753 , -0.05359791, ...,  0.04692035,
        -0.35446193,  0.04547449],
       [ 1.82580196,  6.50519909,  0.4502142 , ..., -0.11721229,
        -0.32619859,  0.0980383 ],
       [ 3.89118706,  6.29366203, -0.07248868, ...,  0.08781199,
        -0.5235218 ,  0.25706967],
       ...,
       [ 3.94341851,  7.5267248 ,  0.17735372, ..., -0.0247584 ,
        -0.42046698,  0.16597895],
       [ 4.62712858,  6.46675171, -0.12189381, ...,  0.03958338,
        -0.37031281,  0.16457751],
       [ 3.61406705,  7.40974494,  0.02875608, ..., -0.0835595 ,
        -0.39089874,  0.08897803]])

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Define the pipeline (scaling + KNN)
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Define hyperparameter grid
knn_param_grid = {
    'knn__n_neighbors': [3, 5, 7, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__p': [1, 2],  # Minkowski p=1 (manhattan), p=2 (euclidean)
}

# Grid search with stratified 15-fold CV
knn_grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    cv=15,
    scoring='accuracy',
    n_jobs=3,
    verbose=1
)

# Fit grid search
knn_grid_search.fit(X_train, y_train)

# Report results
print("=== Best KNN Parameters ===")
print(knn_grid_search.best_params_)
print(f"Best Mean CV Accuracy: {knn_grid_search.best_score_:.4f}")

# Evaluate on best fold's split (you can reuse your fold logic if needed)
best_knn_model = knn_grid_search.best_estimator_
y_pred = best_knn_model.predict(X_test)
print("\n=== Test Set Performance ===")
print(classification_report(y_test, y_pred, target_names=["Control", "Alzheimer's"]))

Fitting 15 folds for each of 40 candidates, totalling 600 fits
=== Best KNN Parameters ===
{'knn__metric': 'euclidean', 'knn__n_neighbors': 3, 'knn__p': 1, 'knn__weights': 'distance'}
Best Mean CV Accuracy: 0.9270

=== Test Set Performance ===
              precision    recall  f1-score   support

     Control       0.67      0.62      0.64      5552
 Alzheimer's       0.58      0.64      0.61      4624

    accuracy                           0.63     10176
   macro avg       0.63      0.63      0.63     10176
weighted avg       0.63      0.63      0.63     10176



In [32]:
from sklearn.preprocessing import StandardScaler

# making sure min-maxed ! also might change results a little 
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Define the pipeline (scaling + KNN)
knn_pipeline = Pipeline([
    ('knn', KNeighborsClassifier())
])

# Define hyperparameter grid
knn_param_grid = {
    'knn__n_neighbors': [3, 5, 7, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__p': [1, 2],  # Minkowski p=1 (manhattan), p=2 (euclidean)
}

# Grid search with stratified 15-fold CV
knn_grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    cv=15,
    scoring='accuracy',
    n_jobs=3,
    verbose=1
)

# Fit grid search
knn_grid_search.fit(X_train_scaled, y_train)

# Report results
print("=== Best KNN Parameters ===")
print(knn_grid_search.best_params_)
print(f"Best Mean CV Accuracy: {knn_grid_search.best_score_:.4f}")

# Evaluate on best fold's split (you can reuse your fold logic if needed)
best_knn_model = knn_grid_search.best_estimator_
y_pred = best_knn_model.predict(X_test_scaled)
print("\n=== Test Set Performance ===")
print(classification_report(y_test, y_pred, target_names=["Control", "Alzheimer's"]))

Fitting 15 folds for each of 40 candidates, totalling 600 fits
=== Best KNN Parameters ===
{'knn__metric': 'euclidean', 'knn__n_neighbors': 3, 'knn__p': 1, 'knn__weights': 'distance'}
Best Mean CV Accuracy: 0.9269

=== Test Set Performance ===
              precision    recall  f1-score   support

     Control       0.67      0.62      0.64      5552
 Alzheimer's       0.58      0.64      0.61      4624

    accuracy                           0.63     10176
   macro avg       0.63      0.63      0.63     10176
weighted avg       0.63      0.63      0.63     10176



In [ ]:
%%time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

# Step 1: Scale once
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Define hyperparameter grid
k_values = [3, 5, 7, 9, 11]
weights_list = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan']
p_values = [1, 2]  # Only used if metric is 'minkowski'

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]
print("start")
# Step 4: Manual hyperparameter search
for k in k_values:
    for weight in weights_list:
        for metric in metrics:
            for p in p_values:

                if metric != 'minkowski' and p != 2:
                    continue  # p is irrelevant unless using 'minkowski'

                label = f"KNN k={k}, weight={weight}, metric={metric}, p={p}"
                model = KNeighborsClassifier(
                    n_neighbors=k,
                    weights=weight,
                    metric=metric if metric != 'minkowski' else 'minkowski',
                    p=p
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


start

=== Cross-Validation: KNN k=3, weight=uniform, metric=euclidean, p=2 ===
Mean Accuracy: 0.9114
Std Deviation: 0.0025
All Fold Scores: [0.9091 0.9122 0.9149 0.9081 0.9125]


In [37]:
%%time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: No need to scale for trees
X_train_tree = X_train
X_test_tree = X_test

# Step 2: More complex hyperparameter grid
max_depths = [10, 15, 20, None]  # None = fully grow the tree
min_samples_splits = [2, 3, 5]   # Smaller split thresholds
min_samples_leafs = [1, 2]       # Smaller leaves allowed

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for depth in max_depths:
    for min_split in min_samples_splits:
        for min_leaf in min_samples_leafs:

            label = f"DecisionTree max_depth={depth}, min_split={min_split}, min_leaf={min_leaf}"
            model = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=min_split,
                min_samples_leaf=min_leaf,
                max_features=None,  # use all features
                ccp_alpha=0.0,      # disable pruning
                random_state=42
            )

            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            # Step 5: Cross-validation scores
            scores = cross_val_score(model, X_train_tree, y_train, cv=5, scoring='accuracy', n_jobs=3)
            mean_acc, std_acc = scores.mean(), scores.std()
            print(f"Mean Accuracy: {mean_acc:.4f}")
            print(f"Std Deviation: {std_acc:.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            # Step 6: Best fold evaluation
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            best_fold_index = np.argmax(scores)
            for i, (train_idx, test_idx) in enumerate(skf.split(X_train_tree, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train_tree[train_idx], X_train_tree[test_idx]
                    y_tr, y_te = y_train[train_idx], y_train[test_idx]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    train_acc = accuracy_score(y_tr, y_pred_train)
                    test_acc = accuracy_score(y_te, y_pred_test)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))

                    results.append((label, mean_acc, std_acc, train_acc, test_acc))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: DecisionTree max_depth=10, min_split=2, min_leaf=1 ===
Mean Accuracy: 0.7565
Std Deviation: 0.0036
All Fold Scores: [0.755  0.7609 0.7544 0.7517 0.7603]

=== Best Fold Summary: DecisionTree max_depth=10, min_split=2, min_leaf=1 ===
Train Accuracy: 0.7805
Test Accuracy: 0.7570
              precision    recall  f1-score   support

     Control       0.80      0.62      0.69     15123
 Alzheimer's       0.74      0.87      0.80     18608

    accuracy                           0.76     33731
   macro avg       0.77      0.74      0.75     33731
weighted avg       0.76      0.76      0.75     33731

⏱️ Duration: 8.3s

=== Cross-Validation: DecisionTree max_depth=10, min_split=2, min_leaf=2 ===
Mean Accuracy: 0.7563
Std Deviation: 0.0037
All Fold Scores: [0.755  0.7607 0.754  0.7513 0.7607]

=== Best Fold Summary: DecisionTree max_depth=10, min_split=2, min_leaf=2 ===
Train Accuracy: 0.7803
Test Accuracy: 0.7569
              precision    recall  f1-score   support



In [34]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np


# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    "KNN-tuned": KNeighborsClassifier(
    n_neighbors=3,
    weights='distance',
    metric='euclidean',
    p=1
    ),
    "KNN": KNeighborsClassifier(),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),

    # Classic SVM (can be slow on big data)
    "SVM": make_pipeline(
        # StandardScaler(), 
        SVC(kernel='linear',
        probability=True)
    )
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break



=== Cross-Validation: KNN-tuned ===
Mean Accuracy: 0.8348
Standard Deviation: 0.0061
All Fold Scores: [0.8314 0.8261 0.8362 0.8487 0.8306 0.8367 0.83   0.8355 0.8395 0.8308
 0.835  0.8256 0.8395 0.8435 0.8325]

=== Best Fold Summary: KNN-tuned ===
Train Accuracy: 1.0000
Test Accuracy: 0.8402
              precision    recall  f1-score   support

     Control       0.82      0.82      0.82      5041
 Alzheimer's       0.85      0.86      0.86      6203

    accuracy                           0.84     11244
   macro avg       0.84      0.84      0.84     11244
weighted avg       0.84      0.84      0.84     11244


=== Cross-Validation: KNN ===
Mean Accuracy: 0.8276
Standard Deviation: 0.0063
All Fold Scores: [0.8284 0.8189 0.8262 0.8388 0.8212 0.8333 0.8206 0.8304 0.8363 0.8218
 0.8297 0.8183 0.8306 0.8347 0.8246]

=== Best Fold Summary: KNN ===
Train Accuracy: 0.8989
Test Accuracy: 0.8308
              precision    recall  f1-score   support

     Control       0.82      0.80      0.8

/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/adm

KeyboardInterrupt: 

# ML experiments and tuning

Notes and concerns:
 Are we folding correclty, it looks like we are retraining the best fold, ??? super weird
 Also seems like we need to do more iterations for SVM ... we can get some more out of it 

 Since hte neural nets that havve best aucuracy almost have 1000% in training, should try and do regurlization to try and 'even' out the accuracy between the two

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
import numpy as np
import time


import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress ConvergenceWarnings only
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Set up test space
kernels = ['linear', 'rbf', 'sigmoid']
Cs = [0.1, 1, 10]
gammas = ['scale', 0.01]  # for rbf/sigmoid
n_estimators = 10
max_iter = 10000

# Replace with your target class names
target_names = ["Control", "Alzheimer's"]

# Loop over kernel/C/gamma
for kernel in kernels:
    for C in Cs:
        if kernel == 'linear':
            label = f"BaggedSVM - linear, C={C}"
            base_model = SVC(kernel=kernel, C=C, probability=False, max_iter=max_iter)
            model = make_pipeline(
                BaggingClassifier(
                    estimator=base_model,
                    n_estimators=n_estimators,
                    max_samples=0.1,
                    n_jobs=3,
                    bootstrap=False,
                    random_state=42
                )
            )
            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
            print(f"Mean Accuracy: {scores.mean():.4f}")
            print(f"Standard Deviation: {scores.std():.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            best_fold_index = np.argmax(scores)
            skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
            for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train[train_index], X_train[test_index]
                    y_tr, y_te = y_train[train_index], y_train[test_index]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    test_acc = accuracy_score(y_te, y_pred_test)
                    train_acc = accuracy_score(y_tr, y_pred_train)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f} sec")

        elif kernel in ['rbf', 'sigmoid']:
            for gamma in gammas:
                label = f"BaggedSVM - {kernel}, C={C}, gamma={gamma}"
                base_model = SVC(kernel=kernel, C=C, gamma=gamma, probability=False, max_iter=max_iter)
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=base_model,
                        n_estimators=n_estimators,
                        max_samples=0.1,
                        n_jobs=3,
                        bootstrap=False,
                        random_state=42
                    )
                )
                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
                print(f"Mean Accuracy: {scores.mean():.4f}")
                print(f"Standard Deviation: {scores.std():.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                best_fold_index = np.argmax(scores)
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train[train_index], X_train[test_index]
                        y_tr, y_te = y_train[train_index], y_train[test_index]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        test_acc = accuracy_score(y_te, y_pred_test)
                        train_acc = accuracy_score(y_tr, y_pred_train)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f} sec")


In [56]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once
scaler = MinMaxScaler(feature_range=(-1, 1))  # You can use (0, 1) if you prefer
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Define hyperparameter grids
layer_configs = [(100,), (128,), (128, 64), (256, 128, 64)]
activations = ['relu', 'tanh']
alphas = [1e-4, 1e-3, 1e-2]
early_stopping_options = [True, False]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force loop over hyperparameter combinations
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=10000,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: MLP (100,), act=relu, alpha=0.0001, early_stop=True ===
Mean Accuracy: 0.8408
Std Deviation: 0.0025
All Fold Scores: [0.8429 0.8374 0.8388 0.8407 0.8443]

=== Best Fold Summary: MLP (100,), act=relu, alpha=0.0001, early_stop=True ===
Train Accuracy: 0.8485
Test Accuracy: 0.8426
              precision    recall  f1-score   support

     Control       0.85      0.79      0.82     15123
 Alzheimer's       0.84      0.89      0.86     18607

    accuracy                           0.84     33730
   macro avg       0.84      0.84      0.84     33730
weighted avg       0.84      0.84      0.84     33730

⏱️ Duration: 48.6s

=== Cross-Validation: MLP (100,), act=relu, alpha=0.0001, early_stop=False ===
Mean Accuracy: 0.8518
Std Deviation: 0.0038
All Fold Scores: [0.8523 0.8543 0.8495 0.8461 0.8571]

=== Best Fold Summary: MLP (100,), act=relu, alpha=0.0001, early_stop=False ===
Train Accuracy: 0.8612
Test Accuracy: 0.8525
              precision    recall  f1-score   su

In [39]:
%%time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale features if needed (optional for GB, but useful with continuous variables)
X_train_boost = X_train_scaled  # assuming you already have this
X_test_boost = X_test_scaled

# Step 2: Define hyperparameter grid for more complex boosting
n_estimators_list = [100, 200]
learning_rates = [0.05, 0.1]
max_depths = [3, 5, 7]
min_samples_leafs = [1, 3]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for n_est in n_estimators_list:
    for lr in learning_rates:
        for depth in max_depths:
            for min_leaf in min_samples_leafs:

                label = f"GradBoost n={n_est}, lr={lr}, depth={depth}, min_leaf={min_leaf}"
                model = GradientBoostingClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=depth,
                    min_samples_leaf=min_leaf,
                    random_state=42
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_boost, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_boost, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_boost[train_idx], X_train_boost[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: GradBoost n=100, lr=0.05, depth=3, min_leaf=1 ===
Mean Accuracy: 0.7508
Std Deviation: 0.0031
All Fold Scores: [0.7506 0.7546 0.7459 0.7537 0.7493]

=== Best Fold Summary: GradBoost n=100, lr=0.05, depth=3, min_leaf=1 ===
Train Accuracy: 0.7535
Test Accuracy: 0.7486
              precision    recall  f1-score   support

     Control       0.79      0.60      0.68     15123
 Alzheimer's       0.73      0.87      0.79     18608

    accuracy                           0.75     33731
   macro avg       0.76      0.74      0.74     33731
weighted avg       0.75      0.75      0.74     33731

⏱️ Duration: 231.3s

=== Cross-Validation: GradBoost n=100, lr=0.05, depth=3, min_leaf=3 ===
Mean Accuracy: 0.7508
Std Deviation: 0.0031
All Fold Scores: [0.7506 0.7546 0.7459 0.7537 0.7493]

=== Best Fold Summary: GradBoost n=100, lr=0.05, depth=3, min_leaf=3 ===
Train Accuracy: 0.7535
Test Accuracy: 0.7486
              precision    recall  f1-score   support

     Control      

KeyboardInterrupt: 

In [47]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

import os

# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    # "BaggedSVM": make_pipeline(
    #     StandardScaler(),
    #     BaggingClassifier(
    #         estimator=SVC(kernel='linear', probability=False),
    #         n_estimators=10,
    #         max_samples=0.1,
    #         n_jobs=3,
    #         bootstrap=False,
    #         random_state=42
    #     )
    # ),
    # Classic SVM (can be slow on big data)
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

os.system('say "SVM is done!"')


=== Cross-Validation: SVM ===
Mean Accuracy: 0.7340
Standard Deviation: 0.0114
All Fold Scores: [0.7266 0.7593 0.7191 0.7315 0.7339 0.7402 0.7543 0.7235 0.7298 0.7247
 0.7302 0.7479 0.725  0.7372 0.7273]

=== Best Fold Summary: SVM ===
Train Accuracy: 0.7350
Test Accuracy: 0.7288
              precision    recall  f1-score   support

     Control       0.80      0.53      0.64      5041
 Alzheimer's       0.70      0.89      0.78      6203

    accuracy                           0.73     11244
   macro avg       0.75      0.71      0.71     11244
weighted avg       0.74      0.73      0.72     11244



0

# ML Testing/tuning # best for nets, tanh 200 hidden layers

In [30]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN": KNeighborsClassifier(),
    # "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(200,), activation='tanh', max_iter=10000, random_state=42)
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break



=== Cross-Validation: NeuralNet ===
Mean Accuracy: 0.8028
Standard Deviation: 0.0213
All Fold Scores: [0.8129 0.7898 0.8265 0.8026 0.7618 0.7954 0.7842 0.8026 0.8273 0.8042
 0.8145 0.7786 0.8106 0.8481 0.7824]

=== Best Fold Summary: NeuralNet ===
Train Accuracy: 0.8155
Test Accuracy: 0.8153
              precision    recall  f1-score   support

     Control       0.89      0.67      0.76       561
 Alzheimer's       0.78      0.93      0.85       690

    accuracy                           0.82      1251
   macro avg       0.83      0.80      0.81      1251
weighted avg       0.83      0.82      0.81      1251



In [ ]:
model = DecisionTreeClassifier(
    max_depth=8,
    max_features=None,
    min_samples_leaf=10,
    min_samples_split=5,
    random_state=42
)

print(f"\n=== Cross-Validation: Regularized DecisionTree ===")

# Cross-validation scores
scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)

print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")
print(f"All Fold Scores: {np.round(scores, 4)}")

# Find the best-performing fold
best_fold_index = np.argmax(scores)

# Evaluate best fold manually
skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    if i == best_fold_index:
        X_tr, X_te = X_train[train_index], X_train[test_index]
        y_tr, y_te = y_train[train_index], y_train[test_index]

        model.fit(X_tr, y_tr)

        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)

        train_acc = accuracy_score(y_tr, y_pred_train)
        test_acc = accuracy_score(y_te, y_pred_test)

        print(f"\n=== Best Fold Summary: Regularized DecisionTree ===")
        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
        break


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [4, 6, 8],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2', None]
}

tree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(tree, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)